# Notebook 2 — Actor Curation

This notebook helps editorial moderators audit and improve actor records in the SSH Open Marketplace.

| # | Section |
|---|---|
| 0 | Setup — imports, API connection, data load |
| 1 | Actors with multiple names (comma in name field) |
| 2 | Orphaned actors — not linked to any item |
| 3 | Duplicate actors — find, compare, merge |
| 4 | Actor website URL status |
| 5 | External ID cleanup |

> **Safety:** All write-back operations (delete, merge) require `DEBUG: False` in `config.yaml`.  
> With `DEBUG: True` (the default) API mutation calls are skipped silently.

## 0. Setup

In [ ]:
import re
import pandas as pd
from datetime import datetime

from sshmarketplacelib import MPData as mpd
from sshmarketplacelib import eval as eva, helper as hel

In [ ]:
mpdata = mpd()
utils  = hel.Util()
check  = eva.URLCheck()

### Load items and save snapshot

`getMPItems` downloads each item category and caches it locally as a `.pickle` file.  
Set the second argument to `True` to reuse the cached copy.

The combined JSON snapshot written below is required by `Util` methods such as `getContributors()` and `getDuplicatedActorsWithItems()`.

In [ ]:
df_tools     = mpdata.getMPItems("toolsandservices",  True)
df_pubs      = mpdata.getMPItems("publications",      True)
df_training  = mpdata.getMPItems("trainingmaterials", True)
df_workflows = mpdata.getMPItems("workflows",         True)
df_datasets  = mpdata.getMPItems("datasets",          True)

# Save combined snapshot — required by Util helper methods
_snapshot_df   = pd.concat([df_tools, df_pubs, df_training, df_workflows, df_datasets], ignore_index=True)
_snapshot_path = f"data/full_items_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
_snapshot_df.to_json(_snapshot_path, orient="records")
print(f"Snapshot saved: {_snapshot_path}  ({len(_snapshot_df):,} items)")

### Load actors

Actors are always downloaded fresh from the live API — there is no local cache for them.

In [ ]:
df_actors = mpdata.getMPItems("actors", False)
print(f"{len(df_actors):,} actors  |  columns: {list(df_actors.columns)}")
df_actors.head(3)

## 1. Actors with multiple names

A comma in the `actor.name` field often means that several people were entered as a single actor entry.  
The cells below build a deduplicated list (one row per actor–item pair) and export it to CSV for manual review.

In [ ]:
contributors = utils.getContributors()

# One row per (actor, item) pair — avoids inflating counts when an actor has multiple roles
contributors_nd = contributors.drop_duplicates(
    subset=['actor.name', 'persistentId'], keep='first', ignore_index=True
)
comma_actors = contributors_nd[contributors_nd['actor.name'].str.contains(",", na=False)]
print(f"{len(comma_actors)} actor–item links with a comma in the actor name")
comma_actors.head(10)

In [ ]:
comma_actors.sort_values('persistentId').to_csv('data/commaactors.csv', index=False)
print("Saved → data/commaactors.csv")

## 2. Orphaned actors

An orphaned actor exists in the Marketplace but is not credited on any item.  
The workflow runs in three steps:

1. **Snapshot cross-reference** — actors absent from the local contributor list are *candidates* for orphan status.
2. **Live API verification** — each candidate is checked via `GET /api/actors/{id}?items=true` to rule out stale snapshots.
3. **Delete** — confirmed orphans are permanently removed.

### Step 1 — Snapshot cross-reference

In [ ]:
contributors = utils.getContributors()   # reads the local snapshot
actor_ids_with_items = set(contributors['actor.id'].dropna().astype(int))

candidates = df_actors[~df_actors['id'].isin(actor_ids_with_items)].copy().reset_index(drop=True)

print(f"Total actors (API):           {len(df_actors):>6,}")
print(f"Actors with items (snapshot): {len(actor_ids_with_items):>6,}")
print(f"Candidates (not in snapshot): {len(candidates):>6,}")
candidates[['id', 'name', 'website']].head(10)

### Step 2 — Live API verification

Calls `GET /api/actors/{id}?items=true` for each candidate.  
This may take several minutes when there are many candidates.

In [ ]:
candidate_ids = candidates['id'].dropna().astype(int).tolist()
print(f"Verifying {len(candidate_ids)} candidates …")

verified = {}
for i, actor_id in enumerate(candidate_ids, 1):
    items_df = mpdata.getItemsforActor(str(actor_id))
    verified[actor_id] = (
        not items_df.empty if isinstance(items_df, pd.DataFrame) else None
    )
    if i % 50 == 0:
        print(f"  {i}/{len(candidate_ids)}")

confirmed_orphan_ids = [aid for aid, has_items in verified.items() if has_items is False]
uncertain_ids        = [aid for aid, has_items in verified.items() if has_items is None]
print(f"\nConfirmed orphans: {len(confirmed_orphan_ids)}")
print(f"Uncertain (API error): {len(uncertain_ids)}")

In [ ]:
confirmed_orphans_df = (
    candidates[candidates['id'].isin(confirmed_orphan_ids)]
    [['id', 'name', 'website']]
    .reset_index(drop=True)
)
confirmed_orphans_df

### Step 3 — Delete confirmed orphans

Review `confirmed_orphans_df` above before running this cell.  
Requires `DEBUG: False` in `config.yaml`.  
The API uses `force=false` — actors affiliated with other actors will be refused and left intact.

In [ ]:
delete_results = []
for actor_id in confirmed_orphan_ids:
    result = mpdata.deleteItem('actors', str(actor_id), force=False)
    ok     = hasattr(result, 'status_code') and result.status_code in (200, 204)
    status = 'deleted' if ok else str(result)
    delete_results.append({'id': actor_id, 'status': status})
    print(f"Actor {actor_id}: {status}")

pd.DataFrame(delete_results)

## 3. Duplicate actors

Identifies actors that appear more than once in the Marketplace, typically sharing the same name.  
The workflow: find duplicate groups → inspect a group → check which duplicates have items → merge.

### 3.1 Find duplicates by name

In [ ]:
filter_attribute = 'name'
df_actor_duplicates = utils.getDuplicates(df_actors, filter_attribute)
print(f"{df_actor_duplicates.shape[0]} rows across all duplicate-name groups")
df_actor_duplicates.sort_values('name').head(10)

### 3.2 Inspect a duplicate group

Set `inspect_ids` to the actor IDs you want to compare.  
Rows where values differ are highlighted in yellow.

In [ ]:
inspect_ids = [3209, 1664]   # <-- set actor IDs here

css_equal = "font-size:1.2rem; border:2px solid silver; background:white; padding:8px 16px"
css_diff  = "font-size:1.2rem; border:2px solid silver; background:lightyellow; padding:8px 16px"

compare_df = df_actor_duplicates[df_actor_duplicates['id'].isin(inspect_ids)]
compare_df.T.style.apply(
    lambda x: [css_equal if len(utils.lists_to_list(x.values)) == 1 else css_diff for _ in x],
    axis=1
)

### 3.3 Duplicate actors with associated items

`getDuplicatedActorsWithItems` cross-references the duplicate-name groups against the item snapshot and returns:
- `df_dup_items` — one row per (actor, item) link
- `df_dup_summary` — one row per actor with a list of linked items

In [ ]:
df_dup_items, df_dup_summary = utils.getDuplicatedActorsWithItems(df_actor_duplicates, filter_attribute)
print(f"{df_dup_summary['name'].nunique()} name groups have at least one associated item")
df_dup_summary.sort_values('name').head(10)

In [ ]:
df_dup_items.head(10)

In [ ]:
df_dup_summary.sort_values('name').to_csv('data/duplicatedactorswithitems.csv', index=False)
print("Saved → data/duplicatedactorswithitems.csv")

### 3.4 Merge duplicated actors

Three steps:
1. Set `keep_id` (actor to keep) and `merge_ids` (actors to absorb into it).
2. Preview which items are currently linked to each actor.
3. Execute the merge — requires `DEBUG: False` in `config.yaml`.  
   The actors in `merge_ids` are deleted after the merge.

In [ ]:
# Step 1 — choose actors
keep_id   = 2266    # actor to KEEP
merge_ids = [3209]  # actor(s) to merge INTO keep_id

compare_df = df_actor_duplicates[df_actor_duplicates['id'].isin([keep_id] + merge_ids)]
compare_df[['id', 'name', 'website', 'affiliations', 'externalIds']].T

In [ ]:
# Step 2 — preview linked items
for aid in [keep_id] + merge_ids:
    items = mpdata.getItemsforActor(str(aid))
    name  = df_actor_duplicates.loc[df_actor_duplicates['id'] == aid, 'name'].values
    label = name[0] if len(name) else str(aid)
    print(f"\nActor {aid} ({label}) — {len(items)} item(s):")
    if not items.empty:
        print(items[['persistentId', 'category', 'label']].to_string(index=False))

In [ ]:
# Step 3 — execute merge (requires DEBUG: False in config.yaml)
print(f"Merging {merge_ids} → {keep_id} …")
result = mpdata.postMergedActors(str(keep_id), ','.join(str(i) for i in merge_ids))
print(f"Done: {result}")

## 4. Actor website URL status

Checks the HTTP status of the `actor.website` field for all actors credited on items.  
Actors with broken URLs (e.g. 404) are exported to CSV for correction.

In [ ]:
contributors = utils.getContributors()
urls_df      = check.checkURLValuesInDataset(contributors, 'actor.website')
urls_df      = urls_df.drop_duplicates()
print(f"{len(urls_df)} unique actor website URLs checked")

In [ ]:
broken_urls = urls_df[urls_df['status'] == 404].sort_values('category')
print(f"{len(broken_urls)} actors with a 404 website URL")
broken_urls.style.format({'MPUrl': utils.make_clickable})

In [ ]:
broken_urls.to_csv('data/urlactors.csv', index=False)
print("Saved → data/urlactors.csv")

## 5. External ID cleanup

Some actors have identifiers that store the full URL (e.g. `https://orcid.org/0000-0001-2345-6789`) rather than the bare ID (`0000-0001-2345-6789`). This section detects those entries so they can be corrected via the API.

In [ ]:
ID_TEMPLATES = [
    {"code": "ORCID",    "urlTemplate": "https://orcid.org/{source-actor-id}"},
    {"code": "DBLP",     "urlTemplate": "https://dblp.org/pid/{source-actor-id}"},
    {"code": "Wikidata", "urlTemplate": "https://www.wikidata.org/wiki/{source-actor-id}"},
    {"code": "GitHub",   "urlTemplate": "https://github.com/{source-actor-id}"},
    {"code": "Twitter",  "urlTemplate": "https://twitter.com/{source-actor-id}"},
    {"code": "ROR",      "urlTemplate": "https://ror.org/{source-actor-id}"},
    {"code": "VIAF",     "urlTemplate": "http://viaf.org/viaf/{source-actor-id}"},
    {"code": "ISNI",     "urlTemplate": "https://isni.org/isni/{source-actor-id}"},
]


def _strip_url_prefix(identifier: str, templates: list) -> tuple:
    """Return (cleaned_id, changed). Strips the URL prefix if the identifier matches a template."""
    for tmpl in templates:
        url = tmpl.get("urlTemplate", "")
        if not url:
            continue
        pattern = re.escape(url).replace(r"\{source\-actor\-id\}", r"(.+)")
        m = re.match(pattern, identifier)
        if m:
            return m.group(1), True
    return identifier, False


def process_external_ids(external_ids: list, templates: list) -> tuple:
    """Clean a list of external-id dicts. Returns (updated_ids, any_changed)."""
    updated, changed = [], False
    for eid in external_ids:
        cleaned, flag = _strip_url_prefix(eid.get("identifier", ""), templates)
        if flag:
            changed = True
        updated.append({**eid, "identifier": cleaned, "changed": flag})
    return updated, changed

In [ ]:
df_actors_ext = df_actors.copy()
df_actors_ext[['_cleaned_ids', '_ids_changed']] = df_actors_ext['externalIds'].apply(
    lambda ids: pd.Series(process_external_ids(ids if isinstance(ids, list) else [], ID_TEMPLATES))
)

changed_df = df_actors_ext[df_actors_ext['_ids_changed']][['name', 'externalIds', '_cleaned_ids']].copy()
print(f"{len(changed_df)} actors have full-URL external identifiers")
changed_df.head(20)